# Iterated MIMIC SMOTE synthesis

Load a dataset, an iterated MIMIC model, and top-level embeddings. Interpolate two neighboring top-level embeddings, decode through all MIMIC levels, and display the resynthesized image.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "vision" / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

ROOT_SRC = PROJECT_ROOT / "src"
VISION_SRC = PROJECT_ROOT / "vision" / "src"
for src_dir in [str(ROOT_SRC), str(VISION_SRC)]:
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

from mimic import IteratedMIMIC
from mimic_vision import (
    load_serialized_vision_dataset,
    load_serialized_vision_embedding,
    plot_smote_synthesis,
    resolve_artifact_file,
    select_embedding_neighbors,
    synthesize_smote_image,
)

In [ ]:
DATASET_FILE = "last"
EMBEDDING_FILE = "last"
MODEL_FILE = "last"

DATASET_DIR = PROJECT_ROOT / "vision" / "data" / "serialized"
EMBEDDING_DIR = PROJECT_ROOT / "vision" / "data" / "embeddings"
MODEL_DIR = PROJECT_ROOT / "vision" / "data" / "models"

SOURCE_INDEX = None
NEIGHBOR_K = 5  # Choose the neighbor randomly among the k nearest non-self embeddings.
LAMBDA_VALUE = 0.5
RANDOM_STATE = 0
PLOT_SIZE = (3, 3)  # Per-panel size; three panels produce figsize=(9, 3).

In [ ]:
dataset_path = resolve_artifact_file(DATASET_FILE, input_dir=DATASET_DIR, pattern="*.pkl")
DATASET_FILE = dataset_path.name
dataset = load_serialized_vision_dataset(dataset_path)

embedding_pattern = f"{Path(DATASET_FILE).stem}_mimic-iterated*.pkl" if str(EMBEDDING_FILE).lower() == "last" else "*.pkl"
embedding_path = resolve_artifact_file(EMBEDDING_FILE, input_dir=EMBEDDING_DIR, pattern=embedding_pattern)
EMBEDDING_FILE = embedding_path.name

model_pattern = f"{Path(EMBEDDING_FILE).stem}.joblib" if str(MODEL_FILE).lower() == "last" else "*.joblib"
model_path = resolve_artifact_file(MODEL_FILE, input_dir=MODEL_DIR, pattern=model_pattern)
MODEL_FILE = model_path.name

embedding_artifact = load_serialized_vision_embedding(embedding_path)
model = IteratedMIMIC.load(model_path)

if embedding_artifact.dataset_file != DATASET_FILE:
    raise ValueError(f"Embedding artifact was created for {embedding_artifact.dataset_file}, not {DATASET_FILE}.")

DATASET_FILE, EMBEDDING_FILE, MODEL_FILE, dataset.X.shape, embedding_artifact.embeddings.shape

In [ ]:
source_index, neighbor_index = select_embedding_neighbors(
    embedding_artifact.embeddings,
    source_index=SOURCE_INDEX,
    k=NEIGHBOR_K,
    random_state=RANDOM_STATE,
)
source_index, neighbor_index

In [ ]:
synthesis = synthesize_smote_image(
    model,
    embedding_artifact.embeddings,
    source_index=source_index,
    neighbor_index=neighbor_index,
    image_shape=dataset.image_shape,
    lambda_value=LAMBDA_VALUE,
)
synthesis.vector.shape, synthesis.image.shape

In [ ]:
fig, axes = plot_smote_synthesis(
    dataset.images,
    synthesis,
    labels=dataset.y,
    target_names=dataset.target_names,
    size=PLOT_SIZE,
)